# Proyecto 3: Reconocimiento de ASL — Versión secuencial

Juan Diego Solís Martínez - 23720  
Víctor Manuel Pérez Chávez - 23731

Solución del Laboratorio 3 de Data Science en un solo flujo: análisis exploratorio, preprocesamiento, dos CNN, una red fully-connected, un modelo clásico y la prueba con fotos propias. Cada fase se cronometra en CPU como línea base de la versión con CUDA.

| Fase | Sección | Dispositivo |
|---|---|---|
| Preparación de datos | 1 a 3 | CPU, un núcleo |
| Entrenamiento e inferencia | 4 a 7 | CPU |
| Resumen de tiempos | 8 | — |

## 0. Configuración

`MODO = "benchmark"` entrena con una submuestra y épocas fijas, y repite las fases medidas `N_REPETICIONES` veces. `MODO = "completo"` reproduce la configuración original del laboratorio en una sola corrida.

In [ ]:
import os
import time

os.environ["CUDA_VISIBLE_DEVICES"] = "-1"  # versión secuencial: solo CPU
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"

MODO = "benchmark"                      # "benchmark" o "completo"
N_REPETICIONES = 3                      # solo aplica en benchmark
IMGS_POR_CLASE_BENCHMARK = 800
EPOCAS_BENCHMARK = 5
N_JOBS = 1                              # 1 = ejecución secuencial
SEED = 42

T_INICIO_NOTEBOOK = time.perf_counter()

In [ ]:
import sys
from pathlib import Path

EN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/JuanDsm04/pdc-final-project.git"

if EN_COLAB and not Path("artifacts").exists():
    !git clone -q {REPO_URL}
    %cd pdc-final-project
    !pip install -q kagglehub

In [ ]:
import re
import json
import random
import platform
import warnings
import subprocess
from collections import Counter
from contextlib import contextmanager

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import psutil
from PIL import Image
from joblib import Parallel, delayed

from skimage.feature import hog
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import (
    classification_report, confusion_matrix, ConfusionMatrixDisplay, accuracy_score
)

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, regularizers
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, Callback

warnings.filterwarnings("ignore")
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

TARGET_SIZE = (64, 64)
N_CLASSES = 29
BATCH_SIZE = 64
REPS = N_REPETICIONES if MODO == "benchmark" else 1

ROOT = Path.cwd()
RESULTS_DIR = ROOT / "results"
FIGURES_DIR = ROOT / "figures"
for carpeta in (RESULTS_DIR, FIGURES_DIR):
    carpeta.mkdir(exist_ok=True)

### 0.1 Hardware y entorno

Se guarda en `results/hardware.json` para documentar en qué máquina se obtuvieron los tiempos.

In [ ]:
def info_hardware():
    info = {
        "fecha": time.strftime("%Y-%m-%d %H:%M:%S"),
        "entorno": "Google Colab" if EN_COLAB else "Local",
        "sistema": f"{platform.system()} {platform.release()}",
        "cpu": platform.processor() or platform.machine(),
        "nucleos_fisicos": psutil.cpu_count(logical=False),
        "nucleos_logicos": psutil.cpu_count(logical=True),
        "ram_gb": round(psutil.virtual_memory().total / 1e9, 1),
        "python": platform.python_version(),
        "tensorflow": tf.__version__,
        "numpy": np.__version__,
        "dispositivos_tf": [d.name for d in tf.config.list_physical_devices()],
        "modo": MODO,
        "repeticiones": REPS,
        "imgs_por_clase_benchmark": IMGS_POR_CLASE_BENCHMARK,
        "epocas_benchmark": EPOCAS_BENCHMARK,
        "n_jobs": N_JOBS,
    }
    cpuinfo = Path("/proc/cpuinfo")
    if cpuinfo.exists():
        modelos = re.findall(r"model name\s*:\s*(.+)", cpuinfo.read_text())
        if modelos:
            info["cpu"] = modelos[0]
    try:
        salida = subprocess.run(
            ["nvidia-smi", "--query-gpu=name,memory.total,driver_version", "--format=csv,noheader"],
            capture_output=True, text=True, timeout=10,
        ).stdout.strip()
        info["gpu"] = salida or "No detectada"
    except (FileNotFoundError, subprocess.TimeoutExpired):
        info["gpu"] = "No detectada"
    return info

HARDWARE = info_hardware()
(RESULTS_DIR / "hardware.json").write_text(json.dumps(HARDWARE, indent=2, ensure_ascii=False))
pd.Series(HARDWARE, dtype=object).to_frame("valor")

### 0.2 Cronometraje

Cada medición se agrega a `results/tiempos_secuencial.csv` en cuanto termina, de modo que no se pierde si la ejecución se interrumpe.

In [ ]:
REGISTRO_TIEMPOS = []
CSV_TIEMPOS = RESULTS_DIR / "tiempos_secuencial.csv"

def registrar(fase, modelo, repeticion, segundos, n_imagenes=None, epoca=None):
    REGISTRO_TIEMPOS.append({
        "parte": "secuencial", "modo": MODO, "fase": fase, "modelo": modelo,
        "repeticion": repeticion, "epoca": epoca, "segundos": segundos,
        "n_imagenes": n_imagenes, "dispositivo": "CPU",
    })
    pd.DataFrame(REGISTRO_TIEMPOS).to_csv(CSV_TIEMPOS, index=False)

@contextmanager
def cronometro(fase, modelo="-", repeticion=1, n_imagenes=None):
    t0 = time.perf_counter()
    yield
    segundos = time.perf_counter() - t0
    registrar(fase, modelo, repeticion, segundos, n_imagenes)
    print(f"[{fase} | {modelo} | rep {repeticion}] {segundos:.2f} s")

class TiempoPorEpoca(Callback):
    def __init__(self, modelo, repeticion=1):
        super().__init__()
        self.modelo, self.repeticion = modelo, repeticion

    def on_epoch_begin(self, epoch, logs=None):
        self.t0 = time.perf_counter()

    def on_epoch_end(self, epoch, logs=None):
        registrar("epoca", self.modelo, self.repeticion, time.perf_counter() - self.t0, epoca=epoch + 1)

## 1. Datos y análisis exploratorio

Dataset ASL Alphabet de Kaggle. Se usa `./archive` si existe; si no, se descarga con `kagglehub`.

In [ ]:
def localizar_dataset():
    local = ROOT / "archive"
    if local.exists():
        return local
    import kagglehub
    return Path(kagglehub.dataset_download("grassknoted/asl-alphabet"))

DATA_DIR = localizar_dataset()
TRAIN_DIR = DATA_DIR / "asl_alphabet_train"
if (TRAIN_DIR / "asl_alphabet_train").is_dir():
    TRAIN_DIR = TRAIN_DIR / "asl_alphabet_train"

print("TRAIN_DIR:", TRAIN_DIR, "-> existe:", TRAIN_DIR.exists())

### 1.1 Formato y balance de clases

In [ ]:
with cronometro("inventario"):
    CLASSES = sorted(d.name for d in TRAIN_DIR.iterdir() if d.is_dir())
    files_per_class = {cls: sorted((TRAIN_DIR / cls).iterdir()) for cls in CLASSES}

conteo = pd.Series({cls: len(files) for cls, files in files_per_class.items()})
extensiones = Counter(f.suffix.lower() for files in files_per_class.values() for f in files)

print(f"{len(CLASSES)} clases: {CLASSES}")
print(f"Total: {conteo.sum():,} imágenes | extensiones: {dict(extensiones)}")
print(f"Imágenes por clase: mínimo {conteo.min():,}, máximo {conteo.max():,}")

fig, ax = plt.subplots(figsize=(14, 5))
ax.bar(conteo.index, conteo.values, color="#4C72B0")
ax.set_title("Distribución de imágenes por clase — conjunto de entrenamiento")
ax.set_xlabel("Clase")
ax.set_ylabel("Número de imágenes")
plt.xticks(rotation=90)
plt.tight_layout()
plt.show()

Las 29 clases tienen 3,000 imágenes JPG cada una: el dataset está balanceado.

### 1.2 Ejemplos y variabilidad dentro de una clase

In [ ]:
rng = random.Random(SEED)
muestras = {cls: rng.sample(files, 8) for cls, files in files_per_class.items()}

n_cols = 6
n_rows = int(np.ceil(len(CLASSES) / n_cols))
fig, axes = plt.subplots(n_rows, n_cols, figsize=(n_cols * 2.2, n_rows * 2.2))
axes = axes.flatten()
for ax, cls in zip(axes, CLASSES):
    ax.imshow(Image.open(muestras[cls][0]))
    ax.set_title(cls, fontsize=10)
for ax in axes:
    ax.axis("off")
plt.suptitle("Un ejemplo por clase", y=1.02, fontsize=14)
plt.tight_layout()
plt.show()

LETRA_EJEMPLO = "A"
fig, axes = plt.subplots(2, 4, figsize=(14.4, 4.5))
for ax, img_path in zip(axes.flatten(), muestras[LETRA_EJEMPLO]):
    ax.imshow(Image.open(img_path))
    ax.axis("off")
plt.suptitle(f"Variabilidad dentro de la clase '{LETRA_EJEMPLO}'", y=1.03, fontsize=13)
plt.tight_layout()
plt.show()

Fondo, iluminación y persona constantes: dentro de una clase solo cambian ligeramente la posición y la rotación de la mano.

### 1.3 Resolución y modo de color

In [ ]:
resoluciones, modos = Counter(), Counter()
for paths in muestras.values():
    for img_path in paths:
        with Image.open(img_path) as img:
            resoluciones[img.size] += 1
            modos[img.mode] += 1

print(f"Muestra de {sum(resoluciones.values())} imágenes")
print("Resoluciones (ancho, alto):", dict(resoluciones))
print("Modos de color:", dict(modos))

### 1.4 Letras visualmente similares

In [ ]:
GRUPOS = {"M/N/S": ["M", "N", "S"], "U/V/R": ["U", "V", "R"], "K/P": ["K", "P"]}

for nombre, letras in GRUPOS.items():
    fig, axes = plt.subplots(len(letras), 4, figsize=(8.8, len(letras) * 2.2))
    for fila, letra in enumerate(letras):
        for col in range(4):
            axes[fila, col].imshow(Image.open(muestras[letra][col]))
            axes[fila, col].axis("off")
        axes[fila, 0].set_title(letra, loc="left", fontsize=12)
    plt.suptitle(f"Grupo {nombre}", fontsize=13)
    plt.tight_layout()
    plt.show()

Las letras de cada grupo solo difieren en la posición de uno o dos dedos, por lo que son las que más probablemente se confundan.

## 2. Partición train / validación / test

Partición estratificada 70/15/15 con semilla 42, generada en el laboratorio y guardada en `artifacts/splits.csv` para que sea idéntica en cualquier sistema operativo. Si el archivo no existe, se regenera con el mismo procedimiento.

In [ ]:
SPLITS_CSV = ROOT / "artifacts" / "splits.csv"

def generar_particion():
    df_all = pd.DataFrame(
        [{"rel": f"{cls}/{f.name}", "clase": cls} for cls, files in files_per_class.items() for f in files]
    )
    train, temp = train_test_split(df_all, test_size=0.30, stratify=df_all["clase"], random_state=SEED)
    val, test = train_test_split(temp, test_size=0.50, stratify=temp["clase"], random_state=SEED)
    return pd.concat([train.assign(split="train"), val.assign(split="val"), test.assign(split="test")])

with cronometro("particion"):
    splits_df = pd.read_csv(SPLITS_CSV) if SPLITS_CSV.exists() else generar_particion()
    splits_df["path"] = [str(TRAIN_DIR / rel) for rel in splits_df["rel"]]
    train_df, val_df, test_df = (
        splits_df.loc[splits_df["split"] == s, ["path", "clase"]].reset_index(drop=True)
        for s in ("train", "val", "test")
    )

label_map = {cls: i for i, cls in enumerate(CLASSES)}
idx_to_class = {i: cls for cls, i in label_map.items()}

print(f"Train: {len(train_df):,} | Val: {len(val_df):,} | Test: {len(test_df):,}")
pd.DataFrame({
    "train": train_df["clase"].value_counts(),
    "val": val_df["clase"].value_counts(),
    "test": test_df["clase"].value_counts(),
}).T

### 2.1 Conjunto de entrenamiento según el modo

En `benchmark` se toma una submuestra estratificada de `IMGS_POR_CLASE_BENCHMARK` imágenes por clase. Validación y test se usan completos en ambos modos.

In [ ]:
if MODO == "benchmark":
    train_fit_df = (
        train_df.groupby("clase", group_keys=False)
        .sample(n=IMGS_POR_CLASE_BENCHMARK, random_state=SEED)
        .reset_index(drop=True)
    )
else:
    train_fit_df = train_df

print(f"Modo: {MODO} | imágenes de entrenamiento: {len(train_fit_df):,} | repeticiones: {REPS}")

## 3. Preprocesamiento de imágenes

Pipeline por imagen: lectura → RGB → resize bicúbico de 200×200 a 64×64 → `float32` → normalización a [0, 1]. Se conserva el color, porque el contraste entre la piel y el fondo verde ayuda a separar la mano.

In [ ]:
def preprocess_image(path, size=TARGET_SIZE):
    with Image.open(path) as img:
        img = img.convert("RGB").resize(size, Image.BICUBIC)
    return np.array(img, dtype=np.float32) / 255.0

ejemplos = [files_per_class[cls][0] for cls in ["A", "B", "V", "M", "del"]]
fig, axes = plt.subplots(2, len(ejemplos), figsize=(len(ejemplos) * 3, 6))
for col, path in enumerate(ejemplos):
    axes[0, col].imshow(Image.open(path))
    axes[0, col].set_title(f"{path.parent.name} — 200×200", fontsize=9)
    axes[1, col].imshow(preprocess_image(path))
    axes[1, col].set_title(f"{path.parent.name} — 64×64", fontsize=9)
    axes[0, col].axis("off")
    axes[1, col].axis("off")
fig.suptitle("Original vs. preprocesada", fontsize=13)
plt.tight_layout()
plt.show()

arr = preprocess_image(ejemplos[0])
print(f"Salida: {arr.shape}, {arr.dtype}, rango [{arr.min():.3f}, {arr.max():.3f}]")

### 3.1 Generadores de Keras

Como en el laboratorio, el entrenamiento lee y preprocesa cada lote desde disco en cada época con `flow_from_dataframe`.

In [ ]:
datagen = ImageDataGenerator(rescale=1.0 / 255.0)

def make_gen(df, shuffle):
    return datagen.flow_from_dataframe(
        dataframe=df, x_col="path", y_col="clase", target_size=TARGET_SIZE,
        color_mode="rgb", class_mode="categorical", batch_size=BATCH_SIZE,
        shuffle=shuffle, seed=SEED, interpolation="bicubic",
    )

train_gen = make_gen(train_fit_df, shuffle=True)
val_gen = make_gen(val_df, shuffle=False)
test_gen = make_gen(test_df, shuffle=False)

print(f"Lotes por época -> train: {len(train_gen)} | val: {len(val_gen)} | test: {len(test_gen)}")

### 3.2 Fase de preparación medida: carga secuencial a memoria

Preprocesa en un solo núcleo todas las imágenes de entrenamiento del modo actual hacia un arreglo `float32` reservado de antemano (unos 1.1 GB en benchmark y 3.0 GB en completo). Esta es la fase que la versión paralela distribuye entre los núcleos del CPU.

In [ ]:
def load_split_as_arrays(df, size=TARGET_SIZE):
    X = np.empty((len(df), *size, 3), dtype=np.float32)
    for i, path in enumerate(df["path"]):
        X[i] = preprocess_image(path, size=size)
    return X, df["clase"].map(label_map).to_numpy(dtype=np.int32)

for rep in range(1, REPS + 1):
    X_prep = y_prep = None
    with cronometro("preprocesamiento", "secuencial", rep, len(train_fit_df)):
        X_prep, y_prep = load_split_as_arrays(train_fit_df)

print(f"X: {X_prep.shape} ({X_prep.nbytes / 1e9:.2f} GB) | y: {y_prep.shape}")
del X_prep, y_prep

## 4. Modelos de Deep Learning (CNN)

En `benchmark` se entrena con épocas fijas y sin EarlyStopping, para que todas las repeticiones hagan el mismo trabajo. En `completo` se usan las épocas máximas y los callbacks del laboratorio.

### 4.1 Funciones de entrenamiento y evaluación

In [ ]:
EPOCAS_LAB = {"cnn_shallow": 30, "cnn_medium": 40, "fc_base": 40}

def epocas(nombre):
    return EPOCAS_LAB[nombre] if MODO == "completo" else EPOCAS_BENCHMARK

def get_callbacks(nombre, repeticion=1):
    callbacks = [
        TiempoPorEpoca(nombre, repeticion),
        ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=4, min_lr=1e-6, verbose=1),
    ]
    if MODO == "completo":
        callbacks.append(EarlyStopping(monitor="val_accuracy", patience=8, restore_best_weights=True, verbose=1))
    return callbacks

def entrenar(model, nombre, repeticion=1, verbose=1):
    with cronometro("entrenamiento", nombre, repeticion, len(train_fit_df)):
        return model.fit(train_gen, epochs=epocas(nombre), validation_data=val_gen,
                         callbacks=get_callbacks(nombre, repeticion), verbose=verbose)

def predecir(model, gen, fase=None, nombre="-", repeticion=1):
    gen.reset()
    if fase is None:
        return model.predict(gen, verbose=0).argmax(axis=1)
    with cronometro(fase, nombre, repeticion, gen.n):
        return model.predict(gen, verbose=0).argmax(axis=1)

def plot_history(history, title):
    fig, axes = plt.subplots(1, 2, figsize=(13, 4))
    for ax, metrica, etiqueta in [(axes[0], "accuracy", "Accuracy"), (axes[1], "loss", "Loss")]:
        ax.plot(history.history[metrica], label="Train", linewidth=2)
        ax.plot(history.history[f"val_{metrica}"], label="Val", linewidth=2)
        ax.set_title(f"{title} — {etiqueta} por época")
        ax.set_xlabel("Época")
        ax.set_ylabel(etiqueta)
        ax.legend()
        ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

### 4.2 Modelo 1: CNN Shallow (2 bloques convolucionales)

In [ ]:
def build_cnn_shallow(input_shape=(64, 64, 3), n_classes=N_CLASSES):
    inputs = keras.Input(shape=input_shape)
    x = inputs
    for filtros in [32, 64]:
        x = layers.Conv2D(filtros, (3, 3), padding="same", activation="relu")(x)
        x = layers.BatchNormalization()(x)
        x = layers.MaxPooling2D((2, 2))(x)
    x = layers.Flatten()(x)
    x = layers.Dense(256, activation="relu")(x)
    x = layers.Dropout(0.5)(x)
    outputs = layers.Dense(n_classes, activation="softmax")(x)
    return keras.Model(inputs, outputs, name="CNN_Shallow")

keras.utils.set_random_seed(SEED)
model1 = build_cnn_shallow()
model1.compile(optimizer=keras.optimizers.Adam(learning_rate=1e-3),
               loss="categorical_crossentropy", metrics=["accuracy"])
model1.summary()

hist1 = entrenar(model1, "cnn_shallow")
plot_history(hist1, "CNN Shallow")
acc1_val = accuracy_score(val_gen.classes, predecir(model1, val_gen))
print(f"CNN Shallow — accuracy en validación: {acc1_val:.4f}")

### 4.3 Modelo 2: CNN Medium (4 bloques, L2 + Dropout)

Es el modelo de referencia del benchmark: se entrena y se evalúa en test `REPS` veces, siempre desde los mismos pesos iniciales.

In [ ]:
def build_cnn_medium(input_shape=(64, 64, 3), n_classes=N_CLASSES, l2_reg=1e-4):
    inputs = keras.Input(shape=input_shape)
    x = inputs
    for i, filtros in enumerate([32, 64, 128, 256], start=1):
        x = layers.Conv2D(filtros, (3, 3), padding="same", activation="relu")(x)
        x = layers.BatchNormalization()(x)
        x = layers.MaxPooling2D((2, 2))(x) if i < 4 else layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(512, activation="relu", kernel_regularizer=regularizers.l2(l2_reg))(x)
    x = layers.Dropout(0.5)(x)
    outputs = layers.Dense(n_classes, activation="softmax")(x)
    return keras.Model(inputs, outputs, name="CNN_Medium")

for rep in range(1, REPS + 1):
    keras.utils.set_random_seed(SEED)
    model2 = build_cnn_medium()
    model2.compile(optimizer=keras.optimizers.Adam(learning_rate=5e-4),
                   loss="categorical_crossentropy", metrics=["accuracy"])
    if rep == 1:
        model2.summary()
    hist2 = entrenar(model2, "cnn_medium", rep, verbose=1 if rep == 1 else 2)
    y_test_medium = predecir(model2, test_gen, "inferencia_test", "cnn_medium", rep)

plot_history(hist2, "CNN Medium")
acc2_val = accuracy_score(val_gen.classes, predecir(model2, val_gen))
print(f"CNN Medium — accuracy en validación: {acc2_val:.4f}")

### 4.4 Comparación y selección de la mejor CNN

In [ ]:
print(pd.DataFrame([
    {"Modelo": "CNN Shallow (2 bloques)", "Val Accuracy": f"{acc1_val:.4f}",
     "Parámetros": f"{model1.count_params():,}", "Épocas": len(hist1.history["loss"])},
    {"Modelo": "CNN Medium (4 bloques)", "Val Accuracy": f"{acc2_val:.4f}",
     "Parámetros": f"{model2.count_params():,}", "Épocas": len(hist2.history["loss"])},
]).to_string(index=False))

if acc2_val >= acc1_val:
    best_name, best_model, y_test_cnn = "CNN Medium", model2, y_test_medium
else:
    best_name, best_model = "CNN Shallow", model1
    y_test_cnn = predecir(model1, test_gen, "inferencia_test", "cnn_shallow")
BEST_KEY = best_name.lower().replace(" ", "_")
print(f"\nMejor CNN: {best_name}")

## 5. Red neuronal fully-connected

Aplana la imagen a 12,288 valores, sin convoluciones. Pierde la estructura espacial y sirve como contraste frente a la CNN.

In [ ]:
def build_fc_base(input_shape=(64, 64, 3), n_classes=N_CLASSES):
    inputs = keras.Input(shape=input_shape)
    x = layers.Flatten()(inputs)
    for unidades, dropout in [(512, 0.5), (256, 0.4)]:
        x = layers.Dense(unidades, activation="relu")(x)
        x = layers.BatchNormalization()(x)
        x = layers.Dropout(dropout)(x)
    outputs = layers.Dense(n_classes, activation="softmax")(x)
    return keras.Model(inputs, outputs, name="FC_Base")

keras.utils.set_random_seed(SEED)
fc1 = build_fc_base()
fc1.compile(optimizer=keras.optimizers.Adam(learning_rate=1e-3),
            loss="categorical_crossentropy", metrics=["accuracy"])
fc1.summary()

hist_fc1 = entrenar(fc1, "fc_base")
plot_history(hist_fc1, "FC Base")
acc_fc_val = accuracy_score(val_gen.classes, predecir(fc1, val_gen))
y_test_fc = predecir(fc1, test_gen, "inferencia_test", "fc_base")
print(f"FC Base — accuracy en validación: {acc_fc_val:.4f}")

## 6. Modelo clásico: ensambles de árboles

Un modelo clásico no aprende sus propias características, así que se extraen a mano. Se comparan dos representaciones: los píxeles en gris a 32×32 y el descriptor HOG (bordes y forma) más una miniatura de color de 8×8. Con `N_JOBS = 1`, la extracción y el ajuste se ejecutan en un solo núcleo.

In [ ]:
HOG_PARAMS = dict(orientations=9, pixels_per_cell=(8, 8), cells_per_block=(2, 2),
                  block_norm="L2-Hys", feature_vector=True)

def a_arreglo(im, size, modo):
    return np.asarray(im.convert(modo).resize(size, Image.BICUBIC), dtype=np.float32) / 255.0

def feat_raw_gray32(im):
    return a_arreglo(im, (32, 32), "L").ravel()

def feat_hog_color(im):
    h = hog(a_arreglo(im, TARGET_SIZE, "L"), **HOG_PARAMS).astype(np.float32)
    return np.concatenate([h, a_arreglo(im, (8, 8), "RGB").ravel()])

FEATURES = {"raw_gray32": feat_raw_gray32, "hog+color": feat_hog_color}

def extraer(path, feat_fn):
    with Image.open(path) as im:
        return feat_fn(im)

def build_matrix(df, feat_fn):
    X = np.asarray(Parallel(n_jobs=N_JOBS, batch_size=128)(delayed(extraer)(p, feat_fn) for p in df["path"]),
                   dtype=np.float32)
    return X, df["clase"].map(label_map).to_numpy()

def subsample(df, n_per_class):
    idx = df.groupby("clase")["path"].sample(n=n_per_class, random_state=SEED).index
    return df.loc[idx].reset_index(drop=True)

for nombre, fn in FEATURES.items():
    print(f"{nombre:<11} dimensión: {extraer(train_df['path'].iloc[0], fn).shape[0]:,}")

### 6.1 Selección de la familia de modelos

Barrido rápido sobre una submuestra de 150 imágenes por clase para entrenar y 75 para validar.

In [ ]:
def candidatos():
    return {
        "KNN(k=3)": KNeighborsClassifier(n_neighbors=3, n_jobs=N_JOBS),
        "RF(300)": RandomForestClassifier(n_estimators=300, n_jobs=N_JOBS, random_state=SEED),
        "ExtraTrees(300)": ExtraTreesClassifier(n_estimators=300, n_jobs=N_JOBS, random_state=SEED),
        "SVC-RBF": make_pipeline(StandardScaler(), SVC(C=10, gamma="scale", random_state=SEED)),
    }

bench_train, bench_val = subsample(train_df, 150), subsample(val_df, 75)
filas = []
with cronometro("barrido", "clasico", 1, len(bench_train) + len(bench_val)):
    for f_nombre, fn in FEATURES.items():
        Xtr, ytr = build_matrix(bench_train, fn)
        Xva, yva = build_matrix(bench_val, fn)
        for m_nombre, modelo in candidatos().items():
            t0 = time.perf_counter()
            modelo.fit(Xtr, ytr)
            t_fit = time.perf_counter() - t0
            filas.append({"Representación": f_nombre, "Modelo": m_nombre,
                          "Val Accuracy": accuracy_score(yva, modelo.predict(Xva)), "Fit (s)": t_fit})

print(pd.DataFrame(filas).sort_values("Val Accuracy", ascending=False)
      .to_string(index=False, float_format=lambda v: f"{v:.4f}"))

Se usa ExtraTrees: en el laboratorio, los ensambles de árboles superaron a KNN y SVC, y ExtraTrees igualó a Random Forest con menos tiempo de ajuste.

### 6.2 Representación y número de árboles

Con el conjunto de entrenamiento del modo actual se prueban ambas representaciones con 50, 100, 200 y 300 árboles. Se elige la configuración con menos árboles dentro de 0.1 puntos del mejor accuracy, y se entrena de nuevo como modelo final.

In [ ]:
matrices = {}
for nombre, fn in FEATURES.items():
    with cronometro("extraccion_features", nombre, 1, len(train_fit_df) + len(val_df)):
        matrices[nombre] = (*build_matrix(train_fit_df, fn), *build_matrix(val_df, fn))

filas = []
with cronometro("ajuste_hiperparametros", "clasico", 1, len(train_fit_df)):
    for nombre, (Xtr, ytr, Xva, yva) in matrices.items():
        for n in [50, 100, 200, 300]:
            modelo = ExtraTreesClassifier(n_estimators=n, n_jobs=N_JOBS, random_state=SEED)
            t0 = time.perf_counter()
            modelo.fit(Xtr, ytr)
            t_fit = time.perf_counter() - t0
            filas.append({"Representación": nombre, "n_estimators": n,
                          "Val Accuracy": accuracy_score(yva, modelo.predict(Xva)), "Fit (s)": t_fit})
            del modelo

ajuste = pd.DataFrame(filas)
print(ajuste.to_string(index=False, float_format=lambda v: f"{v:.4f}"))

elegibles = ajuste[ajuste["Val Accuracy"] >= ajuste["Val Accuracy"].max() - 0.001]
elegida = elegibles.sort_values(["n_estimators", "Val Accuracy"], ascending=[True, False]).iloc[0]
FEAT_ELEGIDA, N_ESTIMATORS = elegida["Representación"], int(elegida["n_estimators"])
MODELO_ELEGIDO = f"ExtraTrees({N_ESTIMATORS}) + {FEAT_ELEGIDA}"
print(f"\nConfiguración elegida: {MODELO_ELEGIDO} | val accuracy: {elegida['Val Accuracy']:.4f}")

X_train_c, y_train_c = matrices[FEAT_ELEGIDA][:2]
del matrices
modelo_final = ExtraTreesClassifier(n_estimators=N_ESTIMATORS, n_jobs=N_JOBS, random_state=SEED)
with cronometro("entrenamiento", "clasico", 1, len(X_train_c)):
    modelo_final.fit(X_train_c, y_train_c)

### 6.3 Evaluación en test

In [ ]:
with cronometro("extraccion_features_test", "clasico", 1, len(test_df)):
    X_test_c, y_test_c = build_matrix(test_df, FEATURES[FEAT_ELEGIDA])
with cronometro("inferencia_test", "clasico", 1, len(test_df)):
    y_pred_c = modelo_final.predict(X_test_c)
print(f"{MODELO_ELEGIDO} — accuracy en test: {accuracy_score(y_test_c, y_pred_c):.4f}")

## 7. Comparación final y pruebas con señas propias

Se comparan los tres modelos con las predicciones sobre las mismas 13,050 imágenes de test obtenidas en las secciones anteriores.

In [ ]:
tiempos_inf = (pd.DataFrame(REGISTRO_TIEMPOS).query("fase == 'inferencia_test'")
               .groupby("modelo").agg(segundos=("segundos", "mean"), n=("n_imagenes", "first")))
n_nodos = sum(t.tree_.node_count for t in modelo_final.estimators_)

RESULTADOS = {
    "Clásico": ("clasico", y_test_c, y_pred_c, f"{n_nodos:,} nodos"),
    best_name: (BEST_KEY, test_gen.classes, y_test_cnn, f"{best_model.count_params():,} parámetros"),
    "FC Base": ("fc_base", test_gen.classes, y_test_fc, f"{fc1.count_params():,} parámetros"),
}
reportes = {nombre: classification_report(y_true, y_pred, target_names=CLASSES, output_dict=True, zero_division=0)
            for nombre, (_, y_true, y_pred, _) in RESULTADOS.items()}
accs = {nombre: r["accuracy"] for nombre, r in reportes.items()}

print(pd.DataFrame([
    {"Modelo": MODELO_ELEGIDO if nombre == "Clásico" else nombre,
     "Test Accuracy": f"{accs[nombre]:.4f}", "Complejidad": complejidad,
     "Inferencia (ms/img)": f"{1000 * tiempos_inf.loc[clave, 'segundos'] / tiempos_inf.loc[clave, 'n']:.3f}"}
    for nombre, (clave, _, _, complejidad) in RESULTADOS.items()
]).to_string(index=False))

nombres = list(accs)
letras = [l for grupo in GRUPOS.values() for l in grupo]
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].bar(nombres, [accs[n] for n in nombres], color=["#4C72B0", "#55A868", "#C44E52"])
axes[0].set_ylim(min(accs.values()) - 0.05, 1.005)
axes[0].set_title("Accuracy en test")
for i, n in enumerate(nombres):
    axes[0].text(i, accs[n] + 0.002, f"{accs[n]:.4f}", ha="center")
x = np.arange(len(letras))
for i, n in enumerate(nombres):
    axes[1].bar(x + (i - 1) * 0.25, [reportes[n][l]["recall"] for l in letras], 0.25, label=n)
axes[1].set_xticks(x)
axes[1].set_xticklabels(letras)
axes[1].set_title("Recall en letras confundibles")
axes[1].legend()
axes[1].grid(True, axis="y", alpha=0.3)
plt.tight_layout()
plt.show()

### 7.1 Evaluación detallada del mejor modelo

In [ ]:
MEJOR_MODELO = max(accs, key=accs.get)
_, y_true, y_pred, _ = RESULTADOS[MEJOR_MODELO]
print(f"Mejor modelo: {MEJOR_MODELO} (accuracy en test: {accs[MEJOR_MODELO]:.4f})\n")
print(classification_report(y_true, y_pred, target_names=CLASSES, zero_division=0))

fig, ax = plt.subplots(figsize=(16, 14))
ConfusionMatrixDisplay(confusion_matrix(y_true, y_pred), display_labels=CLASSES).plot(
    ax=ax, colorbar=False, xticks_rotation=90, cmap="Blues", values_format="d")
ax.set_title(f"Matriz de confusión en test — {MEJOR_MODELO}", fontsize=14)
plt.tight_layout()
plt.show()

### 7.2 Pruebas con fotos de los integrantes

Las 15 fotos de `fotos_grupo/` siguen el patrón `letra_integrante.jpeg`, con 5 letras por integrante. Se recortan al cuadrado central y se preparan igual que el dataset.

In [ ]:
FOTOS_DIR = ROOT / "fotos_grupo"
patron = re.compile(r"^([A-Za-z]+)_([A-Za-z0-9]+)")
registros = [
    {"ruta": ruta, "letra_real": m.group(1).upper(), "integrante": m.group(2)}
    for ruta in sorted(p for ext in ("*.jpg", "*.jpeg", "*.png") for p in FOTOS_DIR.glob(ext))
    if (m := patron.match(ruta.stem))
]

def recorte_cuadrado(ruta):
    with Image.open(ruta) as im:
        im = im.convert("RGB")
        lado = min(im.size)
        izq, arriba = (im.width - lado) // 2, (im.height - lado) // 2
        return im.crop((izq, arriba, izq + lado, arriba + lado))

recortes = [recorte_cuadrado(r["ruta"]) for r in registros]
if MEJOR_MODELO == "Clásico":
    probs = modelo_final.predict_proba(np.asarray([FEATURES[FEAT_ELEGIDA](im) for im in recortes]))
else:
    X_fotos = np.asarray([a_arreglo(im, TARGET_SIZE, "RGB") for im in recortes])
    probs = (best_model if MEJOR_MODELO == best_name else fc1).predict(X_fotos, verbose=0)

for r, pred, conf in zip(registros, probs.argmax(axis=1), probs.max(axis=1)):
    r.update(prediccion=idx_to_class[pred], confianza=conf, correcto=idx_to_class[pred] == r["letra_real"])

n_correctos = sum(r["correcto"] for r in registros)
print(f"Modelo: {MEJOR_MODELO} | aciertos: {n_correctos}/{len(registros)} ({100 * n_correctos / len(registros):.1f}%)")
pd.DataFrame([{"Archivo": r["ruta"].name, "Integrante": r["integrante"], "Letra real": r["letra_real"],
               "Predicción": r["prediccion"], "Confianza": f"{r['confianza'] * 100:.1f}%",
               "Correcto": "✔" if r["correcto"] else "✘"} for r in registros])

In [ ]:
cols = 5
rows = int(np.ceil(len(registros) / cols))
fig, axes = plt.subplots(rows, cols, figsize=(cols * 2.6, rows * 2.9))
for ax in axes.flatten():
    ax.axis("off")
for ax, r, im in zip(axes.flatten(), registros, recortes):
    ax.imshow(im)
    ax.set_title(f"{r['integrante']} — real: {r['letra_real']}\npred: {r['prediccion']} ({r['confianza'] * 100:.0f}%)",
                 color="green" if r["correcto"] else "red", fontsize=9)
fig.suptitle(f"Pruebas con fotos propias — {MEJOR_MODELO}", fontsize=13)
plt.tight_layout()
plt.show()

## 8. Resumen de tiempos

Media y desviación estándar por fase. Se exporta a `results/resumen_secuencial.csv` y `figures/tiempos_secuencial.png` para la sección 2 del informe.

In [ ]:
registrar("notebook_completo", "-", 1, time.perf_counter() - T_INICIO_NOTEBOOK)
tiempos = pd.DataFrame(REGISTRO_TIEMPOS)

resumen = (tiempos[tiempos["fase"] != "epoca"]
           .groupby(["fase", "modelo"], sort=False)
           .agg(repeticiones=("segundos", "size"), media_s=("segundos", "mean"),
                desv_s=("segundos", "std"), n_imagenes=("n_imagenes", "first"))
           .reset_index())
resumen["ms_por_imagen"] = 1000 * resumen["media_s"] / resumen["n_imagenes"]
resumen.to_csv(RESULTS_DIR / "resumen_secuencial.csv", index=False)

por_epoca = (tiempos[tiempos["fase"] == "epoca"].groupby("modelo")["segundos"]
             .agg(epocas="size", media_s="mean", desv_s="std").reset_index())
por_epoca.to_csv(RESULTS_DIR / "tiempo_por_epoca_secuencial.csv", index=False)

print(resumen.round(3).to_string(index=False))
print()
print(por_epoca.round(3).to_string(index=False))

In [ ]:
graf = resumen[resumen["fase"] != "notebook_completo"].copy()
graf["etiqueta"] = np.where(graf["modelo"] == "-", graf["fase"], graf["fase"] + " · " + graf["modelo"])
graf["desv_s"] = graf["desv_s"].fillna(0)
graf = graf.sort_values("media_s")

fig, ax = plt.subplots(figsize=(10, 0.45 * len(graf) + 1.5))
ax.barh(graf["etiqueta"], graf["media_s"], xerr=graf["desv_s"],
        color="#4C72B0", height=0.6, error_kw={"elinewidth": 1, "capsize": 3})
for y, (v, d) in enumerate(zip(graf["media_s"], graf["desv_s"])):
    ax.text((v + d) * 1.08, y, f"{v:,.2f} s", va="center", fontsize=8)
ax.set_xscale("log")
ax.set_xlim(graf["media_s"].min() * 0.5, (graf["media_s"] + graf["desv_s"]).max() * 4)
ax.set_xlabel("Tiempo medio (s, escala logarítmica)")
ax.set_title(f"Tiempo por fase — versión secuencial en CPU (modo {MODO})")
ax.grid(True, axis="x", alpha=0.3)
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.savefig(FIGURES_DIR / "tiempos_secuencial.png", dpi=150, bbox_inches="tight")
plt.show()

print(f"Tiempo total del notebook: {tiempos.loc[tiempos['fase'] == 'notebook_completo', 'segundos'].iloc[-1] / 60:.1f} min")